# 📦 Notebook 01 — Data Engineering
## Ingestion · Validation · Nettoyage · Pipeline ETL

> **Cours mobilisés :** Data Engineering (Séances 05, 12, 26 Janv., 02 Fév., 27 Fév., 02 Mars 2026)

---

## Objectifs
1. Charger et valider le dataset brut `BaseCLD2026.csv`
2. Comprendre la structure des données (dictionnaire des variables)
3. Nettoyer et transformer les données
4. Créer les variables dérivées (feature engineering)
5. Exporter un dataset propre pour l'analyse

---

## Concepts clés
- **ETL** : Extract → Transform → Load
- **Validation de schéma** : s'assurer que les colonnes attendues sont présentes
- **Feature engineering** : créer de nouvelles variables à partir des données existantes
- **Loi log-normale** : les contaminants environnementaux suivent souvent cette distribution

In [ ]:
# Installation des dépendances (à exécuter une seule fois)
# Si une librairie manque, décommentez et exécutez cette cellule

# import subprocess, sys
# subprocess.check_call([sys.executable, "-m", "pip", "install",
#     "pandas", "numpy", "scipy", "matplotlib", "seaborn", "scikit-learn"])
# Pour la cartographie (optionnel) :
# subprocess.check_call([sys.executable, "-m", "pip", "install", "geopandas", "shapely"])

# ── Vérification rapide des imports disponibles ────────────────────────
import importlib
required = ["pandas", "numpy", "scipy", "matplotlib", "seaborn", "sklearn"]
for pkg in required:
    status = "✅" if importlib.util.find_spec(pkg) else "❌ MANQUANT"
    print(f"{status}  {pkg}")


## 1. 📦 Imports

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Modules du projet
from data_engineering.ingestion import load_raw, validate_schema, quick_report
from data_engineering.cleaning  import clean, CLASS_LABELS, DETECTION_THRESHOLD, REGULATORY_THRESHOLD
from data_engineering.pipeline  import run_pipeline

plt.rcParams['figure.dpi'] = 120
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.4f}'.format)

print('✅ Imports OK')

## 2. 🔍 Extract — Chargement des données brutes

In [ ]:
# Chargement du CSV brut
df_raw = load_raw(Path('../data/raw/BaseCLD2026.csv'))
df_raw.head(3)

In [ ]:
# Validation du schéma
rapport = validate_schema(df_raw)
for k, v in rapport.items():
    print(f'  {k:25s}: {v}')

In [ ]:
# Rapport par colonne : types, valeurs manquantes, cardinalité
quick_report(df_raw)

### 💬 Lecture
Le dataset contient **31 126 observations** et **22 variables** couvrant :
- Des variables **d'identification** : ID, commune, année
- Des variables **environnementales** : type de sol, pluviométrie
- Des **mesures analytiques** : taux de chlordécone, taux 5β-hydro
- Des variables **topographiques** (MNT) : pente, TPI, TRI, rugosité, exposition, ombrage
- Des **coordonnées spatiales** : X, Y (EPSG:5490 — UTM zone 20N)

## 3. 🔧 Transform — Nettoyage et Feature Engineering

In [ ]:
# Nettoyage complet via le module cleaning
df = clean(df_raw)
print(f'Shape après nettoyage : {df.shape}')

In [ ]:
# Vérification des nouvelles variables créées
new_cols = ['DETECTED', 'LOG_CHLD', 'CLASS_CONTAMINATION', 'ABOVE_REG_THRESHOLD']
df[new_cols].describe()

In [ ]:
# Distribution des classes de contamination
class_dist = df['CLASS_CONTAMINATION'].value_counts().sort_index()
print('Distribution des classes :')
for cls, n in class_dist.items():
    print(f'  Classe {int(cls)} — {CLASS_LABELS[int(cls)]}: {n:,} ({n/len(df)*100:.1f}%)')

### 💬 Lecture — Feature Engineering

Les variables dérivées créées :

| Variable | Définition | Intérêt |
|---|---|---|
| `DETECTED` | 1 si CLD > 0.1 mg/kg | Cible binaire pour la classification KNN |
| `LOG_CHLD` | log(1 + Taux_CLD) | Normalise la distribution asymétrique |
| `CLASS_CONTAMINATION` | 0–3 selon le niveau | Cible multi-classe |
| `ABOVE_REG_THRESHOLD` | 1 si CLD > 1.0 mg/kg | Indicateur réglementaire |

La **transformation logarithmique** est fondamentale : les contaminants environnementaux
suivent une loi log-normale car ils résultent de processus multiplicatifs
(dilution, dégradation, accumulation). Sans cette transformation, les modèles
statistiques et ML seraient biaisés par les valeurs extrêmes.

## 4. 💾 Load — Export du dataset nettoyé

In [ ]:
# Export vers data/processed/
out_path = Path('../data/processed/chlordecone_clean.csv')
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False, sep=';')
print(f'✅ Dataset exporté : {out_path}')
print(f'   {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

## 5. 🔄 Pipeline complet (alternative)

Le script `src/data_engineering/pipeline.py` exécute les 3 étapes ETL en une commande :

```bash
python src/data_engineering/pipeline.py
# ou avec chemins personnalisés :
python src/data_engineering/pipeline.py --input data/raw/BaseCLD2026.csv --output data/processed/chlordecone_clean.csv
```